# Baseline ML Model

Establish baseline performance with minimal features, then test if including weather features provides significant improvements.

**Approach:**
1. Baseline: airline + cyclical month encoding
2. Enhanced: baseline + weather features
3. Compare performance to decide if weather features help

**Data split (time-based):**
- Train: 2010-2022
- Validation: 2023
- Test: 2024-2025 (held out)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load data
df = pd.read_csv('../data/processed/ml_training_data_syd_mel.csv')
print(f"Shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

In [ ]:
# Extract month number from date column
df['month_num'] = pd.to_datetime(df['month']).dt.month
df['year'] = df['year'].astype(int)

# Create cyclical month encoding
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)

print("Cyclical encoding sample:")
df[['month_num', 'month_sin', 'month_cos']].drop_duplicates().sort_values('month_num')

In [ ]:
# One-hot encode airline
airline_dummies = pd.get_dummies(df['airline'], prefix='airline')
df = pd.concat([df, airline_dummies], axis=1)

print(f"Airlines encoded: {list(airline_dummies.columns)}")

In [ ]:
# Time-based split
# Excluding 2020-2022 (COVID period)

train_mask = ((df['year'] <= 2017) | (df['year'] == 2024))
val_mask = ((df['year'] == 2018) | (df['year'] == 2023))
test_mask = ((df['year'] == 2019) | (df['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025)     : {test_mask.sum()} samples")
print("")
print("Note: 2020-2022 excluded (COVID period)")

In [ ]:
# Define feature sets
airline_cols = [c for c in df.columns if c.startswith('airline_')]
weather_cols = [c for c in df.columns if c.endswith('_dep') or c.endswith('_arr')]

# Baseline features: airline + cyclical month + sectors_flown
baseline_features = airline_cols + ['month_sin', 'month_cos', 'sectors_flown']

# Enhanced features: baseline + weather
enhanced_features = baseline_features + weather_cols

print(f"Baseline features: {len(baseline_features)}")
print(f"Enhanced features: {len(enhanced_features)} (+{len(weather_cols)} weather)")

In [ ]:
# Prepare data splits
X_train_base = df.loc[train_mask, baseline_features]
X_val_base = df.loc[val_mask, baseline_features]
X_test_base = df.loc[test_mask, baseline_features]

X_train_enh = df.loc[train_mask, enhanced_features]
X_val_enh = df.loc[val_mask, enhanced_features]
X_test_enh = df.loc[test_mask, enhanced_features]

# Targets
y_train_reg = df.loc[train_mask, 'delay_rate']
y_val_reg = df.loc[val_mask, 'delay_rate']
y_test_reg = df.loc[test_mask, 'delay_rate']

y_train_clf = df.loc[train_mask, 'is_high_delay']
y_val_clf = df.loc[val_mask, 'is_high_delay']
y_test_clf = df.loc[test_mask, 'is_high_delay']

# Scale features
scaler_base = StandardScaler()
X_train_base_scaled = scaler_base.fit_transform(X_train_base)
X_val_base_scaled = scaler_base.transform(X_val_base)
X_test_base_scaled = scaler_base.transform(X_test_base)

scaler_enh = StandardScaler()
X_train_enh_scaled = scaler_enh.fit_transform(X_train_enh)
X_val_enh_scaled = scaler_enh.transform(X_val_enh)
X_test_enh_scaled = scaler_enh.transform(X_test_enh)

print("Data split and features scaled.")

## 2. Baseline Model (Airline + Month)

### 2.1 Regression: Predict delay_rate

In [ ]:
# Hyperparameter tuning for Ridge regression
alphas = [0.01, 0.1, 1.0, 10.0]
best_alpha = None
best_val_rmse = float('inf')

print("Ridge Regression - Hyperparameter Tuning (Baseline)")
print("-" * 50)

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_base_scaled, y_train_reg)
    
    y_val_pred = model.predict(X_val_base_scaled)
    val_rmse = np.sqrt(mean_squared_error(y_val_reg, y_val_pred))
    val_r2 = r2_score(y_val_reg, y_val_pred)
    
    print(f"alpha={alpha:6.2f}: Val RMSE={val_rmse:.4f}, R²={val_r2:.4f}")
    
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_alpha = alpha

print(f"\nBest alpha: {best_alpha}")

In [ ]:
# Train final baseline regression model
baseline_reg = Ridge(alpha=best_alpha)
baseline_reg.fit(X_train_base_scaled, y_train_reg)

# Validation performance
y_val_pred_base = baseline_reg.predict(X_val_base_scaled)

baseline_reg_metrics = {
    'RMSE': np.sqrt(mean_squared_error(y_val_reg, y_val_pred_base)),
    'MAE': mean_absolute_error(y_val_reg, y_val_pred_base),
    'R2': r2_score(y_val_reg, y_val_pred_base)
}

print("Baseline Regression (Validation Set):")
for metric, value in baseline_reg_metrics.items():
    print(f"  {metric}: {value:.4f}")

### 2.2 Classification: Predict is_high_delay

In [ ]:
# Hyperparameter tuning for Logistic Regression
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
best_C = None
best_val_f1 = 0

print("Logistic Regression - Hyperparameter Tuning (Baseline)")
print("-" * 50)

for C in C_values:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_train_base_scaled, y_train_clf)
    
    y_val_pred = model.predict(X_val_base_scaled)
    val_f1 = f1_score(y_val_clf, y_val_pred)
    val_acc = accuracy_score(y_val_clf, y_val_pred)
    
    print(f"C={C:6.2f}: Val Accuracy={val_acc:.4f}, F1={val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_C = C

print(f"\nBest C: {best_C}")

In [ ]:
# Train final baseline classification model
baseline_clf = LogisticRegression(C=best_C, max_iter=1000, random_state=42)
baseline_clf.fit(X_train_base_scaled, y_train_clf)

# Validation performance
y_val_pred_base_clf = baseline_clf.predict(X_val_base_scaled)
y_val_proba_base_clf = baseline_clf.predict_proba(X_val_base_scaled)[:, 1]

baseline_clf_metrics = {
    'Accuracy': accuracy_score(y_val_clf, y_val_pred_base_clf),
    'F1': f1_score(y_val_clf, y_val_pred_base_clf),
    'ROC-AUC': roc_auc_score(y_val_clf, y_val_proba_base_clf)
}

print("Baseline Classification (Validation Set):")
for metric, value in baseline_clf_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 3. Weather-Enhanced Model

### 3.1 Enhanced Regression

In [ ]:
# Hyperparameter tuning for enhanced Ridge regression
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
best_alpha_enh = None
best_val_rmse_enh = float('inf')

print("Ridge Regression - Hyperparameter Tuning (Enhanced)")
print("-" * 50)

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_enh_scaled, y_train_reg)
    
    y_val_pred = model.predict(X_val_enh_scaled)
    val_rmse = np.sqrt(mean_squared_error(y_val_reg, y_val_pred))
    val_r2 = r2_score(y_val_reg, y_val_pred)
    
    print(f"alpha={alpha:6.2f}: Val RMSE={val_rmse:.4f}, R²={val_r2:.4f}")
    
    if val_rmse < best_val_rmse_enh:
        best_val_rmse_enh = val_rmse
        best_alpha_enh = alpha

print(f"\nBest alpha: {best_alpha_enh}")

In [ ]:
# Train final enhanced regression model
enhanced_reg = Ridge(alpha=best_alpha_enh)
enhanced_reg.fit(X_train_enh_scaled, y_train_reg)

# Validation performance
y_val_pred_enh = enhanced_reg.predict(X_val_enh_scaled)

enhanced_reg_metrics = {
    'RMSE': np.sqrt(mean_squared_error(y_val_reg, y_val_pred_enh)),
    'MAE': mean_absolute_error(y_val_reg, y_val_pred_enh),
    'R2': r2_score(y_val_reg, y_val_pred_enh)
}

print("Enhanced Regression (Validation Set):")
for metric, value in enhanced_reg_metrics.items():
    print(f"  {metric}: {value:.4f}")

### 3.2 Enhanced Classification

In [ ]:
# Hyperparameter tuning for enhanced Logistic Regression
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
best_C_enh = None
best_val_f1_enh = 0

print("Logistic Regression - Hyperparameter Tuning (Enhanced)")
print("-" * 50)

for C in C_values:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_train_enh_scaled, y_train_clf)
    
    y_val_pred = model.predict(X_val_enh_scaled)
    val_f1 = f1_score(y_val_clf, y_val_pred)
    val_acc = accuracy_score(y_val_clf, y_val_pred)
    
    print(f"C={C:6.2f}: Val Accuracy={val_acc:.4f}, F1={val_f1:.4f}")
    
    if val_f1 > best_val_f1_enh:
        best_val_f1_enh = val_f1
        best_C_enh = C

print(f"\nBest C: {best_C_enh}")

In [ ]:
# Train final enhanced classification model
enhanced_clf = LogisticRegression(C=best_C_enh, max_iter=1000, random_state=42)
enhanced_clf.fit(X_train_enh_scaled, y_train_clf)

# Validation performance
y_val_pred_enh_clf = enhanced_clf.predict(X_val_enh_scaled)
y_val_proba_enh_clf = enhanced_clf.predict_proba(X_val_enh_scaled)[:, 1]

enhanced_clf_metrics = {
    'Accuracy': accuracy_score(y_val_clf, y_val_pred_enh_clf),
    'F1': f1_score(y_val_clf, y_val_pred_enh_clf),
    'ROC-AUC': roc_auc_score(y_val_clf, y_val_proba_enh_clf)
}

print("Enhanced Classification (Validation Set):")
for metric, value in enhanced_clf_metrics.items():
    print(f"  {metric}: {value:.4f}")

### 3.3 Comparison: Baseline vs Enhanced

In [ ]:
# Regression comparison
print("=" * 60)
print("REGRESSION COMPARISON (Validation Set)")
print("=" * 60)
print(f"{'Metric':<10} {'Baseline':>12} {'Enhanced':>12} {'Improvement':>12}")
print("-" * 60)

for metric in ['RMSE', 'MAE', 'R2']:
    base_val = baseline_reg_metrics[metric]
    enh_val = enhanced_reg_metrics[metric]
    if metric == 'R2':
        improvement = enh_val - base_val  # Higher is better
    else:
        improvement = base_val - enh_val  # Lower is better
    print(f"{metric:<10} {base_val:>12.4f} {enh_val:>12.4f} {improvement:>+12.4f}")

In [ ]:
# Classification comparison
print("=" * 60)
print("CLASSIFICATION COMPARISON (Validation Set)")
print("=" * 60)
print(f"{'Metric':<10} {'Baseline':>12} {'Enhanced':>12} {'Improvement':>12}")
print("-" * 60)

for metric in ['Accuracy', 'F1', 'ROC-AUC']:
    base_val = baseline_clf_metrics[metric]
    enh_val = enhanced_clf_metrics[metric]
    improvement = enh_val - base_val  # Higher is better for all
    print(f"{metric:<10} {base_val:>12.4f} {enh_val:>12.4f} {improvement:>+12.4f}")

## 4. Final Evaluation on Test Set

In [ ]:
# Determine which model to use for final evaluation
# Use enhanced if it improves validation metrics, otherwise use baseline

use_enhanced_reg = enhanced_reg_metrics['RMSE'] < baseline_reg_metrics['RMSE']
use_enhanced_clf = enhanced_clf_metrics['F1'] > baseline_clf_metrics['F1']

print(f"Best regression model: {'Enhanced' if use_enhanced_reg else 'Baseline'}")
print(f"Best classification model: {'Enhanced' if use_enhanced_clf else 'Baseline'}")

In [ ]:
# Final regression evaluation on test set
print("=" * 60)
print("FINAL REGRESSION RESULTS (Test Set: 2024-2025)")
print("=" * 60)

# Evaluate both models on test set for comparison
y_test_pred_base_reg = baseline_reg.predict(X_test_base_scaled)
y_test_pred_enh_reg = enhanced_reg.predict(X_test_enh_scaled)

print(f"\n{'Model':<12} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
print("-" * 45)

# Baseline
rmse = np.sqrt(mean_squared_error(y_test_reg, y_test_pred_base_reg))
mae = mean_absolute_error(y_test_reg, y_test_pred_base_reg)
r2 = r2_score(y_test_reg, y_test_pred_base_reg)
print(f"{'Baseline':<12} {rmse:>10.4f} {mae:>10.4f} {r2:>10.4f}")

# Enhanced
rmse = np.sqrt(mean_squared_error(y_test_reg, y_test_pred_enh_reg))
mae = mean_absolute_error(y_test_reg, y_test_pred_enh_reg)
r2 = r2_score(y_test_reg, y_test_pred_enh_reg)
print(f"{'Enhanced':<12} {rmse:>10.4f} {mae:>10.4f} {r2:>10.4f}")

In [ ]:
# Final classification evaluation on test set
print("=" * 60)
print("FINAL CLASSIFICATION RESULTS (Test Set: 2024-2025)")
print("=" * 60)

# Evaluate both models on test set
y_test_pred_base_clf = baseline_clf.predict(X_test_base_scaled)
y_test_proba_base_clf = baseline_clf.predict_proba(X_test_base_scaled)[:, 1]

y_test_pred_enh_clf = enhanced_clf.predict(X_test_enh_scaled)
y_test_proba_enh_clf = enhanced_clf.predict_proba(X_test_enh_scaled)[:, 1]

print(f"\n{'Model':<12} {'Accuracy':>10} {'F1':>10} {'ROC-AUC':>10}")
print("-" * 45)

# Baseline
acc = accuracy_score(y_test_clf, y_test_pred_base_clf)
f1 = f1_score(y_test_clf, y_test_pred_base_clf)
auc = roc_auc_score(y_test_clf, y_test_proba_base_clf)
print(f"{'Baseline':<12} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")

# Enhanced
acc = accuracy_score(y_test_clf, y_test_pred_enh_clf)
f1 = f1_score(y_test_clf, y_test_pred_enh_clf)
auc = roc_auc_score(y_test_clf, y_test_proba_enh_clf)
print(f"{'Enhanced':<12} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")

In [ ]:
# Naive baseline comparison
print("\n" + "=" * 60)
print("NAIVE BASELINE COMPARISON")
print("=" * 60)

# Regression: predict mean
naive_pred_reg = np.full_like(y_test_reg, y_train_reg.mean())
naive_rmse = np.sqrt(mean_squared_error(y_test_reg, naive_pred_reg))
naive_mae = mean_absolute_error(y_test_reg, naive_pred_reg)
print(f"\nRegression - Predict mean ({y_train_reg.mean():.4f}):")
print(f"  RMSE: {naive_rmse:.4f}, MAE: {naive_mae:.4f}")

# Classification: predict majority class
majority_class = y_train_clf.mode()[0]
naive_pred_clf = np.full_like(y_test_clf, majority_class)
naive_acc = accuracy_score(y_test_clf, naive_pred_clf)
print(f"\nClassification - Predict majority class ({majority_class}):")
print(f"  Accuracy: {naive_acc:.4f}")

## 5. Feature Importance Analysis

In [ ]:
# Baseline model coefficients (regression)
coef_df = pd.DataFrame({
    'feature': baseline_features,
    'coefficient': baseline_reg.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print("Baseline Regression - Feature Coefficients:")
print(coef_df.to_string(index=False))

In [ ]:
# Enhanced model - top weather feature coefficients
coef_enh_df = pd.DataFrame({
    'feature': enhanced_features,
    'coefficient': enhanced_reg.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print("Enhanced Regression - Top 15 Feature Coefficients:")
print(coef_enh_df.head(15).to_string(index=False))

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline coefficients
ax = axes[0]
coef_df_sorted = coef_df.sort_values('coefficient')
colors = ['#e74c3c' if x > 0 else '#3498db' for x in coef_df_sorted['coefficient']]
ax.barh(coef_df_sorted['feature'], coef_df_sorted['coefficient'], color=colors)
ax.set_xlabel('Coefficient')
ax.set_title('Baseline Model Coefficients')
ax.axvline(0, color='black', linewidth=0.5)

# Enhanced top coefficients
ax = axes[1]
top_coef = coef_enh_df.head(15).sort_values('coefficient')
colors = ['#e74c3c' if x > 0 else '#3498db' for x in top_coef['coefficient']]
ax.barh(top_coef['feature'], top_coef['coefficient'], color=colors)
ax.set_xlabel('Coefficient')
ax.set_title('Enhanced Model - Top 15 Coefficients')
ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 6. Summary & observations

In [ ]:
# Calculate final summary statistics
print("=" * 70)
print("SUMMARY")
print("=" * 70)

# Test set metrics for final comparison
base_test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_test_pred_base_reg))
enh_test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_test_pred_enh_reg))
base_test_r2 = r2_score(y_test_reg, y_test_pred_base_reg)
enh_test_r2 = r2_score(y_test_reg, y_test_pred_enh_reg)

base_test_f1 = f1_score(y_test_clf, y_test_pred_base_clf)
enh_test_f1 = f1_score(y_test_clf, y_test_pred_enh_clf)
base_test_auc = roc_auc_score(y_test_clf, y_test_proba_base_clf)
enh_test_auc = roc_auc_score(y_test_clf, y_test_proba_enh_clf)

print(f"\n{'REGRESSION (Test Set)':<30}")
print(f"  Baseline R²:  {base_test_r2:.4f}")
print(f"  Enhanced R²:  {enh_test_r2:.4f}")
print(f"  Weather adds: {enh_test_r2 - base_test_r2:+.4f} R²")

print(f"\n{'CLASSIFICATION (Test Set)':<30}")
print(f"  Baseline F1:  {base_test_f1:.4f}")
print(f"  Enhanced F1:  {enh_test_f1:.4f}")
print(f"  Weather adds: {enh_test_f1 - base_test_f1:+.4f} F1")

print(f"\n{'NAIVE BASELINE':<30}")
print(f"  Predict mean RMSE: {naive_rmse:.4f}")
print(f"  Majority class accuracy: {naive_acc:.4f}")

# Verdict
print("\n" + "=" * 70)
print("VERDICT")
print("=" * 70)

if base_test_r2 < 0.1:
    print("\n⚠️  REGRESSION: Poor predictive power (R² < 0.1)")
    print("   The model explains very little variance in delay rates.")
elif base_test_r2 < 0.3:
    print("\n⚡ REGRESSION: Weak predictive power (R² 0.1-0.3)")
    print("   The model has some signal but limited practical utility.")
else:
    print("\n✓  REGRESSION: Moderate predictive power (R² > 0.3)")

if abs(enh_test_r2 - base_test_r2) < 0.02:
    print("\n⚠️  WEATHER FEATURES: No meaningful improvement")
    print("   Adding 42 weather features does not improve predictions.")
else:
    print(f"\n✓  WEATHER FEATURES: Improvement of {enh_test_r2 - base_test_r2:+.4f} R²")

### Observations:

* R² is negative
    - Monthly aggregate delay rates are difficult to predict with available flight and weather features
    - Negative values indicate it is even worse than simply using simple mean (time average) to predict future delays.

* "Enhanced" models offer very little improvements
    - Adding weather features doesn't help much.


## 7. Next step

Based on the observations, new features are needed. Lagged feature is usually a good indicator for time series and it will also allow pre/post-COVID train split stratification. The next step will quickly check if lagged feature (delay rate in the preceeding months) is correlated with the current month delay rate.

See: [4a_test_lagged_features.ipynb](//notebooks/4a_test_lagged_features.ipynb)